In [1]:
import json
from transformers import AutoModelForSeq2SeqLM, M2M100Tokenizer
from tqdm import tqdm

# === File Paths ===
input_path = r"C:\Users\pauli\Downloads\EXIST2025_filtered_5yes.json"
output_path = r"C:\Users\pauli\Downloads\EXIST2025_filtered_5yes_first20_M2M100_translated.json"

# === Load M2M100 Model and Tokenizer ===
model_name = "facebook/m2m100_1.2B"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# === Language Codes ===
lang_en = "en"
lang_es = "es"

# === Translation Function ===
def translate(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_length=512,  # Prevent overly long generation
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# === Load Filtered Entries ===
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# === Only Process First 20 Entries ===
first_20_items = list(data.items())[:20]
translated_data = {}

for key, entry in tqdm(first_20_items, desc="Translating first 20 tweets"):
    try:
        tweet = entry.get("tweet", "")
        translated_es = translate(tweet, lang_en, lang_es)
        translated_back_en = translate(translated_es, lang_es, lang_en)

        translated_data[key] = {
            "tweet": tweet,
            "translated_es": translated_es,
            "translated_back_en": translated_back_en
        }

    except Exception as e:
        print(f"⚠️ Error translating {key}: {e}")
        translated_data[key] = {
            "tweet": tweet,
            "translated_es": "",
            "translated_back_en": ""
        }

# === Save Translations ===
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(translated_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ First 20 tweets translated. Output saved to:\n{output_path}")


c:\Users\pauli\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\pauli\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pauli\.cache\huggingface\hub\models--facebook--m2m100_1.2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this 


✅ First 20 tweets translated. Output saved to:
C:\Users\pauli\Downloads\EXIST2025_filtered_5yes_first20_M2M100_translated.json
